In [ ]:
# 1. SHAP Feature Importance & Value Plots

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN
import warnings

warnings.filterwarnings('ignore')

# Load Data
df = pd.read_csv('heart.csv')
le = LabelEncoder()
for col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
feature_names = X.columns.tolist() # Save feature names for plots

# Split (Seed 369)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balance Data
smt = SMOTETomek(random_state=369)
X_res_lgbm, y_res_lgbm = smt.fit_resample(X_train_scaled, y_train)

ada = ADASYN(random_state=369)
X_res_xgb, y_res_xgb = ada.fit_resample(X_train_scaled, y_train)

# Train SOTA Models (Locked Parameters)
model_lgbm = LGBMClassifier(learning_rate=0.2094, num_leaves=29, random_state=369, verbose=-1)
model_lgbm.fit(X_res_lgbm, y_res_lgbm)

model_xgb = XGBClassifier(learning_rate=0.1562, max_depth=3, use_label_encoder=False, eval_metric='logloss', random_state=369)
model_xgb.fit(X_res_xgb, y_res_xgb)


# DEFINE ENSEMBLE PREDICTION FUNCTION

def ensemble_predict_proba(X_input):
    # Ensure input is numpy array for models
    if isinstance(X_input, pd.DataFrame):
        X_input = X_input.values

    # Get probabilities from both models
    prob_lgbm = model_lgbm.predict_proba(X_input)[:, 1]
    prob_xgb = model_xgb.predict_proba(X_input)[:, 1]

    # Apply AIW-GTO Weights
    final_prob = (prob_lgbm * 0.22) + (prob_xgb * 0.78)
    return final_prob

# COMPUTE SHAP VALUES
print("[INFO] Computing SHAP values (KernelExplainer)...")

# Convert test data back to DataFrame for clearer plots
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Use a background dataset (Seed 369)
background_data = shap.sample(X_test_df, 50, random_state=369)
X_shap_sample = shap.sample(X_test_df, 100, random_state=369)

explainer = shap.KernelExplainer(ensemble_predict_proba, background_data)

# Calculate SHAP values
shap_values = explainer.shap_values(X_shap_sample)

#  GENERATE PLOTS
print("[INFO] Generating SHAP Plots...")

# A. Feature Importance Bar Plot (Global Importance)
plt.figure(figsize=(10, 6), dpi=300)
shap.summary_plot(shap_values, X_shap_sample, plot_type="bar", show=False, color="#FF6F61")
plt.title("Feature Importance (AIW-GTO Ensemble)", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("mean(|SHAP value|)", fontsize=12)
plt.tight_layout()
plt.savefig('SHAP_Feature_Importance_Bar.png', dpi=300)
plt.show()

# B. Beeswarm Summary Plot (Directional Impact)
plt.figure(figsize=(10, 6), dpi=300)
shap.summary_plot(shap_values, X_shap_sample, show=False, cmap="viridis")
plt.title("SHAP Summary Plot: Feature Impact", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('SHAP_Summary_Beeswarm.png', dpi=300)
plt.show()

print("\nProcess Complete.")

In [ ]:
# 2. SHAP Decision Plots

import matplotlib.pyplot as plt
probs_shap = winner_model.predict_proba(X_shap)[:, 1]

# Identify indices for the two cases
# Case A: Heart Disease Presence (Find the patient with Highest Probability)
idx_presence = np.argmax(probs_shap)

# Case B: Heart Disease Absence (Find the patient with Lowest Probability)
idx_absence = np.argmin(probs_shap)

# Handle Expected Value (Base Value)
base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[0]

print(f"Base Value (Average Prediction): {base_value:.4f}")
print(f"Plotting Case 1 (Presence): Index {idx_presence}, Prob: {probs_shap[idx_presence]:.4f}")
print(f"Plotting Case 2 (Absence):  Index {idx_absence}, Prob: {probs_shap[idx_absence]:.4f}")

# GRAPH 1: SHAP Decision Plot for Heart Disease PRESENCE
print("\n--- 1. SHAP Decision Plot: Heart Disease Presence ---")
plt.figure()
shap.decision_plot(
    base_value,
    sv[idx_presence],
    X_shap.iloc[idx_presence],
    feature_names=X_shap.columns.tolist(),
    link='logit',  # Optional: converts log-odds to probabilities (0-1 scale)
    show=False
)
plt.title(f"Decision Plot: Heart Disease Presence (Prob: {probs_shap[idx_presence]:.2f})", fontsize=12)
plt.show()

# GRAPH 2: SHAP Decision Plot for Heart Disease ABSENCE
print("\n--- 2. SHAP Decision Plot: Heart Disease Absence ---")
plt.figure()
shap.decision_plot(
    base_value,
    sv[idx_absence],
    X_shap.iloc[idx_absence],
    feature_names=X_shap.columns.tolist(),
    link='logit',
    show=False
)
plt.title(f"Decision Plot: Heart Disease Absence (Prob: {probs_shap[idx_absence]:.2f})", fontsize=12)
plt.show()